In [44]:
import os
import glob
import pandas as pd

import timm
import pandas as pd
import torchvision.transforms as T

from wildlife_datasets import splits
from wildlife_tools.inference import KnnClassifier
from wildlife_tools.features import DeepFeatures
from wildlife_tools.similarity import CosineSimilarity
from wildlife_tools.data import ImageDataset

In [45]:
root_dir = r'/media/filming/2025-白海豚/20240824_JM_01/'

In [46]:
metainfo = pd.read_csv(root_dir + "FILTERTED_FIN_METAINFO.csv", index_col=0)

In [10]:
model = timm.create_model('hf-hub:BVRA/MegaDescriptor-L-384', pretrained=True)

In [47]:
metainfo

,img_id,path,x_min,x_max,y_min,y_max,orig_img,crop_conf,orig_img_h,orig_img_w,clearness,select
0,0,FIN/1214_20240824JM01ZRA11214_FIN00.JPG,3403,4110,2776,3200,1214_20240824JM01ZRA11214.JPG,0.669546,5760,8640,0.316150,True
1,1,FIN/1215_20240824JM01ZRA11215_FIN00.JPG,3597,4297,2754,3204,1215_20240824JM01ZRA11215.JPG,0.804073,5760,8640,0.671580,True
2,2,FIN/1216_20240824JM01ZRA11216_FIN00.JPG,3695,4441,2774,3226,1216_20240824JM01ZRA11216.JPG,0.791573,5760,8640,0.709832,True
3,3,FIN/1217_20240824JM01ZRA11217_FIN00.JPG,3711,4453,2796,3238,1217_20240824JM01ZRA11217.JPG,0.824041,5760,8640,0.633289,True
4,4,FIN/1218_20240824JM01ZRA11218_FIN00.JPG,3731,4435,2806,3239,1218_20240824JM01ZRA11218.JPG,0.831123,5760,8640,0.262074,True
...,...,...,...,...,...,...,...,...,...,...,...,...
2618,2618,FIN/0057_20240824JM01ZRA10057_FIN00.JPG,1827,2240,1296,1530,0057_20240824JM01ZRA10057.JPG,0.873636,2880,4320,0.997790,True
2623,2623,FIN/0044_20240824JM01ZRA10044_FIN00.JPG,2192,2423,1561,1688,0044_20240824JM01ZRA10044.JPG,0.813805,2880,4320,0.110677,True
2624,2624,FIN/0045_20240824JM01ZRA10045_FIN00.JPG,2170,2434,1533,1677,0045_20240824JM01ZRA10045.JPG,0.835904,2880,4320,0.470295,True
2625,2625,FIN/0046_20240824JM01ZRA10046_FIN00.JPG,2137,2427,1526,1678,0046_20240824JM01ZRA10046.JPG,0.833445,2880,4320,0.618249,True


In [48]:
metainfo['identity'] = metainfo["img_id"]
#metainfo['path'] = root_dir + metainfo['path'] 

In [49]:
transform = T.Compose([
    T.Resize([384, 384]),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])
dataset = ImageDataset(
        root = root_dir,
        metadata=metainfo,
        transform=transform,
)

In [51]:
extractor = DeepFeatures(model, device='cuda', batch_size=32) 
# batch size 32 to match 12G gpu memory
features = extractor(dataset)

100%|███████████████████████████████████████████████████████████████| 54/54 [00:52<00:00,  1.03it/s]


In [52]:
features.save(root_dir + "/FILTERED_deepfeatures")

In [56]:
features.metadata.path = root_dir + "/" +  features.metadata.path 

In [68]:
root_dir = os.path.split("/media/filming/2025-白海豚/20240824_JM_01/FILTERED_deepfeatures")[0]

'/media/filming/2025-白海豚/20240824_JM_01'